# RECLin-PT — notebook único (etapa 7)

Um só notebook, parametrizado, que chama os scripts finos de `codigo/scripts/`
(os mesmos comandos do `codigo/Makefile` e de `codigo/docs/reproducao.md`).
Substitui os cinco notebooks do legado, que não são usados.

Regras:

- **Escopo da fase:** só o BioBERTpt (`pucpr/biobertpt-all`) e a seed 42
  (protocolo `codigo/docs/protocolo/fase-biobertpt-seed42.md`). A célula de
  parâmetros recusa outros valores.
- As seções 1 a 3 só leem partições e execuções verificadas: não treinam nem
  avaliam o TEST.
- A seção 4 (fase científica) treina e avalia o TEST: cada passo só roda com a
  autorização explícita daquela execução (`AUTORIZADO[...] = True`), na ordem do
  protocolo; o TEST de cada sistema é avaliado uma única vez.
- Nenhuma célula usa resultados do legado nem executa comandos de controle de versão.

In [ ]:
# Parâmetros (célula marcada como "parameters" para execução parametrizada)
ENCODER = "biobertpt"
SEED = 42
REVISAO = ""            # commit exato (40 caracteres) do BioBERTpt, definido no passo 0
NATUREZA = "cientifica"  # "validacao" para o relatório da rodada de validação técnica
EXECUCOES = None         # None = codigo/resultados/execucoes
AUTORIZADO = {           # um passo da fase científica só roda com a autorização explícita dele
    "pretreino": False, "verificar_inicializacao": False, "validacao_1v": False,
    "baseline": False, "calibrado_dev": False, "restrito": False, "pair_aware": False,
    "estagio1": False, "estagio2": False, "compor_dev": False, "filtro_dev": False,
    "test_baseline": False, "test_calibrado": False, "test_restrito": False,
    "test_pair_aware": False, "test_two_stage": False, "test_filtro": False,
}

In [ ]:
# Preparação: localiza codigo/ e define como chamar os scripts
import subprocess
import sys
from pathlib import Path

if (ENCODER, SEED) != ("biobertpt", 42):
    raise ValueError("A fase atual usa só BioBERTpt e seed 42 (protocolo da fase); outros valores estão fora do escopo.")

AQUI = Path.cwd()
if (AQUI / "codigo" / "reclin").is_dir():
    CODIGO = AQUI / "codigo"
else:
    CODIGO = next(p for p in [AQUI, *AQUI.parents] if (p / "reclin").is_dir() and (p / "scripts").is_dir())
SCRIPTS = CODIGO / "scripts"
EXEC = Path(EXECUCOES) if EXECUCOES else CODIGO / "resultados" / "execucoes"
RELATORIOS = CODIGO / "resultados" / "relatorios" / NATUREZA
ANALISES = CODIGO / "resultados" / "analises"
MODELO = CODIGO / "modelos_pretreinados" / f"biobertpt-all@{REVISAO}"
SUFIXO = f"{ENCODER}_seed{SEED}"


def rodar(script, *args):
    comando = [sys.executable, str(SCRIPTS / script), *map(str, args)]
    print("$", " ".join(comando))
    r = subprocess.run(comando)
    if r.returncode != 0:
        print(f"(saída {r.returncode})")
    return r.returncode


def autorizado(passo):
    if not AUTORIZADO.get(passo):
        print(f"Passo '{passo}' não autorizado: nada foi executado.")
        return False
    if passo not in ("calibrado_dev", "compor_dev", "filtro_dev") and not passo.startswith("test_")             and len(REVISAO) != 40:
        raise ValueError("Informe REVISAO (o commit de 40 caracteres do BioBERTpt).")
    return True


print("codigo/ =", CODIGO)

## 1. Dados: partições congeladas contra o MANIFEST

In [ ]:
rodar("preparar_dados.py", "conferir")
rodar("analisar.py", "lexico")

## 2. Análises descritivas (partições congeladas)

`max-gap`, `alinhamento` e `marcadores` usam TRAIN e DEV; o TEST só entra com
`--incluir-test`, como descrição.

In [ ]:
rodar("analisar.py", "corpus", "--saida", ANALISES / "corpus.json")
rodar("analisar.py", "max-gap", "--saida", ANALISES / "max_gap.json")
rodar("analisar.py", "alinhamento", "--saida", ANALISES / "alinhamento.json")
rodar("analisar.py", "marcadores", "--saida", ANALISES / "marcadores.json")

## 3. Inventário das execuções do código novo (proveniência)

In [ ]:
rodar("gerar_tcc.py", "inventario", "--execucoes", EXEC)

## 4. Fase científica (etapa 8) — só com autorização explícita de cada passo

A ordem é a do protocolo (seção 6). Cada célula verifica `AUTORIZADO[...]` antes
de rodar qualquer coisa.

In [ ]:
if autorizado("pretreino"):
    rodar("pretreino.py", "baixar", "--revisao", REVISAO)
if autorizado("verificar_inicializacao"):
    rodar("two_stage.py", "verificar-inicializacao", "--modelo", MODELO)
if autorizado("validacao_1v"):
    rodar("treinar.py", "--estrategia", "baseline", "--modelo", MODELO, "--validacao-epocas", 1, "--saida", EXEC)
    rodar("two_stage.py", "treinar", "--estagio", 1, "--modelo", MODELO, "--validacao-epocas", 1, "--saida", EXEC)
    rodar("two_stage.py", "treinar", "--estagio", 2, "--modelo", MODELO, "--validacao-epocas", 2, "--saida", EXEC)
    rodar("two_stage.py", "compor-dev", "--estagio1", EXEC / "twostage_e1_validacao_1ep",
          "--estagio2", EXEC / "twostage_e2_validacao_2ep", "--saida", EXEC)
    rodar("baseline_calibrado.py", "calibrar-dev", "--base", EXEC / "baseline_validacao_1ep", "--saida", EXEC)

In [ ]:
if autorizado("baseline"):
    rodar("treinar.py", "--estrategia", "baseline", "--modelo", MODELO, "--saida", EXEC, "--checkpoint-a-cada", 500)
if autorizado("calibrado_dev"):
    rodar("baseline_calibrado.py", "calibrar-dev", "--base", EXEC / f"baseline_{SUFIXO}", "--saida", EXEC)
if autorizado("filtro_dev"):   # léxico congelado da D4 (min_freq=10, só do TRAIN v2)
    rodar("filtro_pistas.py", "--base-execucao", EXEC / f"baseline_{SUFIXO}", "--particao", "dev", "--saida", EXEC)
if autorizado("restrito"):
    rodar("treinar.py", "--estrategia", "restrito", "--modelo", MODELO, "--saida", EXEC)
if autorizado("pair_aware"):
    rodar("treinar.py", "--estrategia", "pair_aware", "--modelo", MODELO, "--saida", EXEC)
if autorizado("estagio1"):
    rodar("two_stage.py", "treinar", "--estagio", 1, "--modelo", MODELO, "--saida", EXEC, "--checkpoint-a-cada", 500)
if autorizado("estagio2"):
    rodar("two_stage.py", "treinar", "--estagio", 2, "--modelo", MODELO, "--saida", EXEC)
if autorizado("compor_dev"):
    rodar("two_stage.py", "compor-dev", "--estagio1", EXEC / f"twostage_e1_{SUFIXO}",
          "--estagio2", EXEC / f"twostage_e2_{SUFIXO}", "--saida", EXEC)

Antes do TEST: conferência da proveniência e do congelamento (passo 6). O
filtro léxico usa o léxico congelado; o `min_freq=10` dele foi escolhido na pendência D4 (só TRAIN v2)
(`codigo/docs/entregas/etapa-7-pendencias.md`).

In [ ]:
rodar("pretreino.py", "conferir", "--execucoes", *[EXEC / n for n in (
    f"baseline_{SUFIXO}", f"baseline_calibrado_{SUFIXO}", f"restrito_{SUFIXO}", f"pairaware_{SUFIXO}",
    f"twostage_e1_{SUFIXO}", f"twostage_e2_{SUFIXO}", f"twostage_{SUFIXO}", f"filtro_{SUFIXO}")])

In [ ]:
if autorizado("test_baseline"):
    rodar("avaliar_test.py", "--execucao", EXEC / f"baseline_{SUFIXO}")
if autorizado("test_calibrado"):
    rodar("baseline_calibrado.py", "aplicar-test", "--execucao", EXEC / f"baseline_calibrado_{SUFIXO}")
if autorizado("test_restrito"):
    rodar("avaliar_test.py", "--execucao", EXEC / f"restrito_{SUFIXO}")
if autorizado("test_pair_aware"):
    rodar("avaliar_test.py", "--execucao", EXEC / f"pairaware_{SUFIXO}")
if autorizado("test_two_stage"):
    rodar("two_stage.py", "avaliar-test", "--execucao", EXEC / f"twostage_{SUFIXO}")
if autorizado("test_filtro"):  # depois do TEST do baseline
    rodar("filtro_pistas.py", "--base-execucao", EXEC / f"baseline_{SUFIXO}", "--particao", "test", "--saida", EXEC)

## 5. Tabelas, curvas, figuras e relatório

Só execuções verificadas pelo inventário, de uma natureza (`NATUREZA`). O que
ainda não foi executado aparece como *pendente*.

In [ ]:
rodar("gerar_tcc.py", "tudo", "--execucoes", EXEC, "--natureza", NATUREZA, "--destino", RELATORIOS)
relatorio = RELATORIOS / "relatorio.md"
if relatorio.is_file():
    from IPython.display import Markdown, display
    display(Markdown(relatorio.read_text(encoding="utf-8")))

## 6. Conferência do texto do TCC (pendente enquanto a pasta `tcc/` não existir)

In [ ]:
rodar("conferir_tcc.py", "--relatorios", RELATORIOS, "--execucoes", EXEC)